<a href="https://colab.research.google.com/github/TeunDm/API_test/blob/main/report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VitaCall MLOps Platform — Groepsrapport

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SundaeR/vitacall-mlops/blob/main/notebooks/report.ipynb)

**Vak:** Machine Learning Engineering & Operations
**Team:** SundaeR
**Datum:** Mei 2026

---

## Architectuur

```
[Callcenter audio]
        │
        ▼
[Edge Model: Whisper ONNX]   ← spraakherkenning, draait lokaal op de server
        │ transcriptie
        ▼
[Cloud Model: DistilBERT]    ← sentimentanalyse, draait als REST API
        │ urgentie + label
        ▼
[Dashboard teamleiders]
```

| Component | Model | Dataset | Locatie |
|-----------|-------|---------|---------|
| Edge ASR | Whisper-tiny (ONNX INT8) | Mozilla Common Voice / LibriSpeech | Lokale server |
| Cloud NLP | DistilBERT (fine-tuned) | IMDb Reviews (50.000) | Cloud REST API |

**MLOps-componenten:** PySpark ETL · Structured Streaming · Data monitoring · MLflow · FastAPI · ONNX · GitHub Actions CI/CD/CT


## 1. Installatie & configuratie

Alle benodigde packages worden in de eerste cel geïnstalleerd, conform de inleverrichtlijnen.

In [1]:
# Installeer alle benodigde packages
%pip install pyspark "datasets<3.0.0" soundfile librosa \
    transformers scikit-learn torch mlflow \
    fastapi uvicorn pydantic requests tqdm \
    optimum[onnxruntime] -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [2]:
import io
import json
import os
import re
import time
import threading
from datetime import datetime

import librosa
import mlflow
import mlflow.pytorch
import numpy as np
import pandas as pd
import soundfile as sf
from datasets import load_dataset, Audio, Dataset
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoProcessor,
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    Trainer,
    TrainingArguments,
)
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    avg, col, count, current_timestamp, length, udf,
)
from pyspark.sql.functions import min as spark_min, max as spark_max
from pyspark.sql.types import IntegerType, StringType, StructField, StructType

# ── Globale paden (Colab: /content als basis) ─────────────────────────────────
BASE = '/content'
TEXT_RAW_CSV       = f'{BASE}/data/raw/text/imdb.csv'
TEXT_PROCESSED_CSV = f'{BASE}/data/processed/text/imdb_clean.csv'
TEXT_PARQUET       = f'{BASE}/data/processed/text/imdb_spark'
AUDIO_PROCESSED    = f'{BASE}/data/processed/audio'
AUDIO_METADATA_CSV = f'{BASE}/data/processed/audio/metadata.csv'
AUDIO_PARQUET      = f'{BASE}/data/processed/audio/metadata_spark'
MONITORING_DIR     = f'{BASE}/monitoring'
BASELINE_PATH      = f'{BASE}/monitoring/baseline.json'
REPORT_PATH        = f'{BASE}/monitoring/report.json'
MODEL_PATH         = f'{BASE}/models/sentiment'
RANDOM_SEED = 42
print('Imports en configuratie geladen.')

Imports en configuratie geladen.


## 2. Data Ingestion

De data-ingestion bestaat uit drie onderdelen:
1. **Tekst ETL** IMDb-reviews downloaden, schoonmaken en opslaan als Parquet
2. **Audio ETL** Audioclips downloaden, resamplen naar 16 kHz en opslaan als wav
3. **Structured Streaming** Simulatie van een realtime gespreksstroom

Alle pipelines zijn geïmplementeerd met **PySpark** (`local[*]` mode, alle CPU-cores).
Waarom PySpark? Pandas laadt alles in RAM op één machine. Spark verdeelt het werk over cores en schaalt naar een cluster als de data groeit.


### 2.1 Tekst ETL Pipeline

**Extract:** IMDb-dataset via HuggingFace Datasets
**Transform:** UDF verwijdert HTML-tags, leestekens en overbodige witruimte
**Load:** Parquet (efficiënt voor ML-modellen) + CSV (voor monitoring)

Een **UDF (User Defined Function)** is een gewone Python-functie verpakt zodat Spark hem gedistribueerd per rij kan toepassen.
**Lazy evaluation:** Spark bouwt eerst een uitvoeringsplan en voert pas uit bij een actie (`.count()`, `.show()`).

In [3]:
# VitaCall ETL Pipelines
class TextETLPipeline:
    """ETL-pipeline voor de IMDb-tekstdataset (cloud sentiment model)."""

    _RE_HTML  = re.compile(r'<[^>]+>')
    _RE_CHARS = re.compile(r'[^a-z0-9\s]')
    _RE_SPACE = re.compile(r'\s+')

    def __init__(self, spark: SparkSession) -> None:
        self.spark = spark
        self._clean_udf = udf(self._clean_text, StringType())

    @staticmethod
    def _clean_text(text: str) -> str:
        """Verwijder HTML, leestekens en overtollige witruimte."""
        if text is None:
            return ''
        text = text.lower()
        text = TextETLPipeline._RE_HTML.sub('', text)
        text = TextETLPipeline._RE_CHARS.sub('', text)
        return TextETLPipeline._RE_SPACE.sub(' ', text).strip()

    def extract(self) -> None:
        """Stap 1 Download IMDb als nog niet aanwezig."""
        if os.path.exists(TEXT_RAW_CSV):
            print(f'Data al aanwezig: {TEXT_RAW_CSV}')
            return
        print('Downloaden IMDb via HuggingFace')
        dataset = load_dataset('imdb')
        df = pd.concat([
            dataset['train'].to_pandas(),
            dataset['test'].to_pandas(),
        ], ignore_index=True)
        os.makedirs(os.path.dirname(TEXT_RAW_CSV), exist_ok=True)
        df.to_csv(TEXT_RAW_CSV, index=False)
        print(f'Opgeslagen: {len(df)} reviews')

    def transform(self):
        """Stap 2 Laad in Spark, pas UDF toe, filter lege rijen."""
        df = self.spark.read.csv(TEXT_RAW_CSV, header=True, inferSchema=True, multiLine=True, escape='"')
        print(f'Geladen: {df.count()} rijen')
        df_clean = (
            df
            .withColumn('text_clean', self._clean_udf(col('text')))
            .filter(length(col('text_clean')) > 10)
            .select('text_clean', 'label')
        )
        print('\nStatistieken per label:')
        df_clean.groupBy('label').agg(
            count('*').alias('aantal'),
            avg(length('text_clean')).alias('gem_lengte'),
        ).show()
        return df_clean

    def load(self, df_clean) -> None:
        """Stap 3 Sla op als Parquet én CSV."""
        os.makedirs(TEXT_PARQUET, exist_ok=True)
        df_clean.write.mode('overwrite').parquet(TEXT_PARQUET)
        os.makedirs(os.path.dirname(TEXT_PROCESSED_CSV), exist_ok=True)
        df_clean.toPandas().to_csv(TEXT_PROCESSED_CSV, index=False)
        print(f'Parquet: {TEXT_PARQUET}')
        print(f'CSV    : {TEXT_PROCESSED_CSV}')

    def run(self) -> None:
        """Volledige ETL: extract → transform → load."""
        print('Tekst ETL Pipeline')
        self.extract()
        df_clean = self.transform()
        self.load(df_clean)
        print(f'Klaar! {df_clean.count()} reviews verwerkt.')

In [4]:
spark = (
    SparkSession.builder
    .appName('VitaCall-TextETL')
    .master('local[*]')
    .config('spark.ui.enabled', 'false')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')

tekst_pipeline = TextETLPipeline(spark)
tekst_pipeline.run()
spark.stop()

Tekst ETL Pipeline
Downloaden IMDb via HuggingFace


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Opgeslagen: 50000 reviews
Geladen: 50000 rijen

Statistieken per label:
+-----+------+----------+
|label|aantal|gem_lengte|
+-----+------+----------+
|    0| 25000|1226.79468|
|    1| 25000|1260.37496|
+-----+------+----------+

Parquet: /content/data/processed/text/imdb_spark
CSV    : /content/data/processed/text/imdb_clean.csv
Klaar! 50000 reviews verwerkt.


### 2.2 Audio ETL Pipeline

**Extract:** Mozilla Common Voice (of LibriSpeech als fallback) via streaming
**Transform:** Resamplen naar 16 kHz (Whisper-vereiste), stille clips filteren (RMS < 0.01)
**Load:** wav-bestanden opslaan + metadata verrijken met PySpark UDF

Spark verwerkt de *metadata* (een tabel per clip); de ruwe audiosamples worden met `soundfile` verwerkt. Spark is gemaakt voor tabellen en niet voor signaaldata.

In [5]:
# VitaCall ETL Pipelines
class AudioETLPipeline:
    """ETL-pipeline voor de audiodataset (edge ASR-model)."""

    TARGET_SR = 16_000
    MIN_DUR   = 1.0
    MAX_DUR   = 15.0

    def __init__(self, spark: SparkSession, sample_size: int = 100) -> None:
        self.spark       = spark
        self.sample_size = sample_size
        self._cat_udf    = udf(self._categorize_duration, StringType())

    @staticmethod
    def _categorize_duration(d: float) -> str:
        """Deelt een audioclip in naar duurcategorie (kort / middel / lang)."""
        if d is None:
            return 'onbekend'
        if d < 3.0:
            return 'kort'
        return 'middel' if d <= 8.0 else 'lang'

    def _decode_audio(self, sample: dict) -> tuple:
        """Lees audio bytes, converteer naar mono float32 op TARGET_SR."""
        a = sample['audio']
        arr, sr = (sf.read(io.BytesIO(a['bytes'])) if a.get('bytes')
                   else sf.read(a['path']))
        if arr.ndim > 1:
            arr = arr.mean(axis=1)
        if sr != self.TARGET_SR:
            arr = librosa.resample(
                arr.astype(np.float32), orig_sr=sr, target_sr=self.TARGET_SR,
            )
        return arr.astype(np.float32), self.TARGET_SR

    def extract(self) -> pd.DataFrame:
        """Stap 1 Download clips; gebruik LibriSpeech als fallback."""
        os.makedirs(AUDIO_PROCESSED, exist_ok=True)
        existing = [f for f in os.listdir(AUDIO_PROCESSED) if f.endswith('.wav')]
        if len(existing) >= self.sample_size:
            print(f'Audio al aanwezig: {len(existing)} clips')
            return pd.read_csv(AUDIO_METADATA_CSV)

        try:
            print('Downloaden Mozilla Common Voice')
            ds = load_dataset(
                'mozilla-foundation/common_voice_17_0', 'en',
                split='validation', streaming=True,
            )
            key = 'sentence'
        except Exception as exc:
            print(f'Common Voice niet beschikbaar ({exc})\nLibriSpeech gebruiken')
            ds = load_dataset('librispeech_asr', 'clean',
                              split='validation', streaming=True)
            key = 'text'

        ds = ds.cast_column('audio', Audio(decode=False))
        records = []
        for i, s in enumerate(ds):
            if i >= self.sample_size:
                break
            try:
                audio, sr = self._decode_audio(s)
                dur = len(audio) / sr
                if not (self.MIN_DUR <= dur <= self.MAX_DUR):
                    continue
                if np.sqrt(np.mean(audio ** 2)) < 0.01:
                    continue
                fname = f'sample_{i:05d}.wav'
                sf.write(os.path.join(AUDIO_PROCESSED, fname), audio, sr)
                records.append({
                    'filename': fname,
                    'sentence': s.get(key, ''),
                    'duration_sec': round(dur, 3),
                    'sample_rate': sr,
                })
            except Exception:
                continue

        df = pd.DataFrame(records)
        df.to_csv(AUDIO_METADATA_CSV, index=False)
        print(f'{len(df)} clips opgeslagen')
        return df

    def transform(self, df_meta: pd.DataFrame):
        """Stap 2 Verrijk metadata met duurcategorie via PySpark UDF."""
        df = self.spark.createDataFrame(df_meta)
        df_v = df.withColumn('duur_categorie', self._cat_udf(col('duration_sec')))
        print('\nVerdeling per duurcategorie:')
        df_v.groupBy('duur_categorie').agg(
            count('*').alias('aantal'),
            avg('duration_sec').alias('gem_duur_sec'),
            spark_min('duration_sec').alias('min_sec'),
            spark_max('duration_sec').alias('max_sec'),
        ).orderBy('duur_categorie').show()
        return df_v

    def load(self, df_spark) -> None:
        """Stap 3 Sla metadata op als Parquet."""
        os.makedirs(AUDIO_PARQUET, exist_ok=True)
        df_spark.write.mode('overwrite').parquet(AUDIO_PARQUET)
        print(f'Parquet: {AUDIO_PARQUET}')

    def run(self) -> None:
        """Volledige audio ETL-pipeline."""
        print('Audio ETL Pipeline')
        df_meta = self.extract()
        df_spark = self.transform(df_meta)
        self.load(df_spark)
        print(f'\nKlaar! {len(df_meta)} clips | gem. duur: {df_meta["duration_sec"].mean():.2f}s')

In [6]:
spark = (
    SparkSession.builder
    .appName('VitaCall-AudioETL')
    .master('local[*]')
    .config('spark.ui.enabled', 'false')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')

audio_pipeline = AudioETLPipeline(spark, sample_size=100)
audio_pipeline.run()
spark.stop()

Audio ETL Pipeline
Downloaden Mozilla Common Voice


Common Voice niet beschikbaar (The directory at hf://datasets/mozilla-foundation/common_voice_17_0@11dc88355e899d1bf2df74f01b904a8544a17b33 doesn't contain any data files)
LibriSpeech gebruiken


Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

99 clips opgeslagen

Verdeling per duurcategorie:
+--------------+------+-----------------+-------+-------+
|duur_categorie|aantal|     gem_duur_sec|min_sec|max_sec|
+--------------+------+-----------------+-------+-------+
|          kort|    14|2.695357142857143|  1.955|  2.985|
|          lang|     9|9.782777777777778|   8.45|  12.16|
|        middel|    76|4.881644736842105|   3.01|  7.855|
+--------------+------+-----------------+-------+-------+

Parquet: /content/data/processed/audio/metadata_spark

Klaar! 99 clips | gem. duur: 5.02s


### 2.3 Structured Streaming realtime transcriptieverwerking

In productie bij VitaCall komen gesprekstranscripties realtime binnen vanuit het callcenter.
Hier simuleren we dat met een generator die elke 3 seconden een nieuw CSV-bestand schrijft naar een inputmap. Spark pikt dit automatisch op.

**Micro-batch** (gekozen aanpak): verwerkt data in blokjes elke X seconden (latency ~3–5 s). Betrouwbaarder dan continuous processing en voldoende voor VitaCall.
**Lazy evaluation:** Spark bouwt het uitvoeringsplan bij `.readStream`, voert pas uit bij `.start()`.


In [7]:
STREAM_TRANSCRIPTS = [
    ('Ik heb al drie dagen pijn in mijn borst en weet niet wat ik moet doen', 1),
    ('Kunnen jullie mij helpen met het aanvragen van thuiszorg voor mijn moeder', 0),
    ('Ik voel me heel verdrietig en kan niet meer slapen de laatste tijd', 1),
    ('Ik wil graag een afspraak maken voor een terugbelverzoek van de verpleegkundige', 0),
    ('Mijn medicijnen zijn op en de apotheek is gesloten wat moet ik nu doen', 1),
    ('Kan ik informatie krijgen over de dagopvang mogelijkheden in mijn regio', 0),
]

URGENTIE_WOORDEN = {'pijn', 'gevallen', 'borst', 'slapen', 'verdrietig', 'medicijnen', 'nood'}


class StreamingPipeline:
    """Simuleert realtime verwerking van gesprekstranscripties met Structured Streaming."""

    SCHEMA = StructType([
        StructField('tekst', StringType(), True),
        StructField('label', IntegerType(), True),
    ])

    def __init__(self, spark: SparkSession,
                 input_dir: str, checkpoint_dir: str) -> None:
        self.spark          = spark
        self.input_dir      = input_dir
        self.checkpoint_dir = checkpoint_dir
        self._urgentie_udf  = udf(self._detecteer_urgentie, StringType())

    @staticmethod
    def _detecteer_urgentie(tekst: str) -> str:
        """Keyword-gebaseerde urgentiedetectie (productie: vervangen door cloud model)."""
        if tekst is None:
            return 'onbekend'
        return 'urgent' if set(tekst.lower().split()) & URGENTIE_WOORDEN else 'normaal'

    def _schrijf_batches(self, n: int = 6, interval: float = 3.0) -> None:
        """Schrijft elke `interval` seconden een nieuw CSV-bestand naar de inputmap."""
        for i in range(n):
            tekst, label = STREAM_TRANSCRIPTS[i % len(STREAM_TRANSCRIPTS)]
            path = os.path.join(self.input_dir, f'batch_{i:03d}.csv')
            with open(path, 'w', encoding='utf-8') as f:
                f.write('tekst,label\n')
                f.write(f'\"{tekst}\",{label}\n')
            print(f'  [Generator] Batch {i + 1}/{n}: {tekst[:50]}')
            time.sleep(interval)

    def run(self, max_batches: int = 6) -> None:
        """Start stream + generator, wacht op voltooiing."""
        print('Structured Streaming Pipeline')
        for d in [self.input_dir, self.checkpoint_dir]:
            os.makedirs(d, exist_ok=True)

        df_stream = (
            self.spark.readStream
            .format('csv')
            .option('header', 'true')
            .schema(self.SCHEMA)
            .load(self.input_dir)
        )
        df_out = (
            df_stream
            .withColumn('urgentie',    self._urgentie_udf(col('tekst')))
            .withColumn('lengte',      length(col('tekst')))
            .withColumn('tijdstempel', current_timestamp())
            .select('tijdstempel', 'tekst', 'urgentie', 'lengte', 'label')
        )
        query = (
            df_out.writeStream
            .outputMode('append')
            .format('console')
            .option('truncate', False)
            .option('checkpointLocation', self.checkpoint_dir)
            .trigger(processingTime='3 seconds')
            .start()
        )

        generator = threading.Thread(
            target=self._schrijf_batches,
            kwargs={'n': max_batches, 'interval': 3.0},
            daemon=True,
        )
        generator.start()
        generator.join()
        time.sleep(5)
        query.stop()
        print('\nStreaming gestopt.')

In [8]:
spark = (
    SparkSession.builder
    .appName('VitaCall-Streaming')
    .master('local[*]')
    .config('spark.ui.enabled', 'false')
    .config('spark.sql.shuffle.partitions', '2')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')

streaming = StreamingPipeline(
    spark,
    input_dir=f'{BASE}/streaming/input',
    checkpoint_dir=f'{BASE}/streaming/checkpoint',
)
streaming.run(max_batches=6)
spark.stop()

Structured Streaming Pipeline
  [Generator] Batch 1/6: Ik heb al drie dagen pijn in mijn borst en weet ni
  [Generator] Batch 2/6: Kunnen jullie mij helpen met het aanvragen van thu
  [Generator] Batch 3/6: Ik voel me heel verdrietig en kan niet meer slapen
  [Generator] Batch 4/6: Ik wil graag een afspraak maken voor een terugbelv
  [Generator] Batch 5/6: Mijn medicijnen zijn op en de apotheek is gesloten
  [Generator] Batch 6/6: Kan ik informatie krijgen over de dagopvang mogeli

Streaming gestopt.


## 3. Data Monitoring & Drift Detectie

Na elke pipeline-run bewaken we de kwaliteit van de verwerkte data.
**Datadrift** treedt op als de data-verdeling verschuift: nieuwe accenten, veranderend belscript, seizoensgebonden hulpvragen. Een model dat op de oude verdeling is getraind presteert dan slechter.

| Drempel | Actie |
|---------|-------|
| > 10% afwijking | WAARSCHUWING monitor nauwlettend |
| > 20% afwijking | KRITIEK hertraining aanbevolen |

In [9]:
# VitaCall Datamonitor en hertraining
class DataMonitor:
    """Bewaakt datakwaliteit en detecteert drift ten opzichte van een baseline."""

    DRIFT_THRESHOLD   = 0.10  # 10% = waarschuwing
    RETRAIN_THRESHOLD = 0.20  # 20% = hertraining

    def __init__(self, text_csv: str, audio_csv: str,
                 baseline_path: str, report_path: str) -> None:
        self.text_csv      = text_csv
        self.audio_csv     = audio_csv
        self.baseline_path = baseline_path
        self.report_path   = report_path
        os.makedirs(os.path.dirname(baseline_path), exist_ok=True)

    def _compute_text_stats(self) -> dict:
        """Berekent kwaliteitsstatistieken voor de tekstdata."""
        if not os.path.exists(self.text_csv):
            return {}
        df = pd.read_csv(self.text_csv)
        return {
            'row_count':       int(len(df)),
            'avg_text_length': round(float(df['text_clean'].str.len().mean()), 2),
            'label_pos_ratio': round(float((df['label'] == 1).mean()), 4),
            'empty_count':     int(df['text_clean'].isna().sum()),
        }

    def _compute_audio_stats(self) -> dict:
        """Berekent kwaliteitsstatistieken voor de audiodata."""
        if not os.path.exists(self.audio_csv):
            return {}
        df = pd.read_csv(self.audio_csv)
        return {
            'clip_count':       int(len(df)),
            'avg_duration_sec': round(float(df['duration_sec'].mean()), 3),
            'min_duration_sec': round(float(df['duration_sec'].min()), 3),
            'max_duration_sec': round(float(df['duration_sec'].max()), 3),
        }

    def _check_drift(self, baseline: dict, current: dict, section: str) -> list:
        """Vergelijkt huidige statistieken met baseline; retourneert afwijkingen."""
        issues = []
        for key, bval in baseline.items():
            if key not in current or bval == 0:
                continue
            change = abs(current[key] - bval) / abs(bval)
            if change >= self.RETRAIN_THRESHOLD:
                level = 'KRITIEK'
            elif change >= self.DRIFT_THRESHOLD:
                level = 'WAARSCHUWING'
            else:
                continue
            issues.append({
                'level': level, 'section': section, 'metric': key,
                'baseline': bval, 'current': current[key],
                'change_pct': round(change * 100, 1),
            })
        return issues

    def run(self, reset_baseline: bool = False) -> dict:
        """Voer monitoringcheck uit, sla rapport op en retourneer resultaat."""
        print('Data Monitoring Pipeline')
        current = {
            'timestamp': datetime.now().isoformat(),
            'text':  self._compute_text_stats(),
            'audio': self._compute_audio_stats(),
        }
        print('Tekst stats :', current['text'])
        print('Audio stats :', current['audio'])

        if reset_baseline or not os.path.exists(self.baseline_path):
            with open(self.baseline_path, 'w') as f:
                json.dump(current, f, indent=2)
            print('\nBaseline opgeslagen. Dit is de referentie voor toekomstige runs.')
            return {'issues': [], 'retrain_recommended': False}

        with open(self.baseline_path) as f:
            baseline = json.load(f)

        issues = (
            self._check_drift(baseline.get('text', {}),  current['text'],  'tekst') +
            self._check_drift(baseline.get('audio', {}), current['audio'], 'audio')
        )
        retrain = any(i['level'] == 'KRITIEK' for i in issues)

        report = {
            'timestamp':          current['timestamp'],
            'current_stats':      {k: v for k, v in current.items() if k != 'timestamp'},
            'issues':             issues,
            'retrain_recommended': retrain,
        }
        with open(self.report_path, 'w') as f:
            json.dump(report, f, indent=2)

        print('\n Driftcontrole')
        if not issues:
            print('Geen afwijkingen. Data is stabiel.')
        for issue in issues:
            print(f'  [{issue["level"]}] {issue["section"]}.{issue["metric"]}: '
                  f'{issue["baseline"]} -> {issue["current"]} ({issue["change_pct"]}%)')
        print()
        print('HERTRAINING AANBEVOLEN' if retrain else 'Geen hertraining nodig.')
        return report

In [10]:
monitor = DataMonitor(
    text_csv=TEXT_PROCESSED_CSV,
    audio_csv=AUDIO_METADATA_CSV,
    baseline_path=BASELINE_PATH,
    report_path=REPORT_PATH,
)
# Eerste run: sla baseline op
rapport = monitor.run(reset_baseline=True)

# Tweede run: vergelijk met baseline (voer opnieuw uit na nieuwe ETL-run)
rapport = monitor.run()

Data Monitoring Pipeline
Tekst stats : {'row_count': 50000, 'avg_text_length': 1243.58, 'label_pos_ratio': 0.5, 'empty_count': 0}
Audio stats : {'clip_count': 99, 'avg_duration_sec': 5.018, 'min_duration_sec': 1.955, 'max_duration_sec': 12.16}

Baseline opgeslagen. Dit is de referentie voor toekomstige runs.
Data Monitoring Pipeline
Tekst stats : {'row_count': 50000, 'avg_text_length': 1243.58, 'label_pos_ratio': 0.5, 'empty_count': 0}
Audio stats : {'clip_count': 99, 'avg_duration_sec': 5.018, 'min_duration_sec': 1.955, 'max_duration_sec': 12.16}

 Driftcontrole
Geen afwijkingen. Data is stabiel.

Geen hertraining nodig.


## 4. Modelleren & Experiment Tracking

We trainen het **cloud sentiment model**: DistilBERT fine-tuned op IMDb-reviews.

**Transfer learning:** we starten van een voorgetraind DistilBERT-model en trainen alleen de classificatielaag bij op onze data. Dit is sneller dan trainen vanaf nul en vereist minder data.

**MLflow** logt elke run automatisch:
- *Parameters*: hyperparameters (epochs, batch size, learning rate)
- *Metrics*: F1-score en accuracy per epoch
- *Artifacts*: het getrainde model en het classificatierapport

**Vereiste (productvereistendocument):** F1-score > 0.85 op de testset.

In [11]:
def laad_imdb_data(processed_csv: str, raw_csv: str) -> pd.DataFrame:
    """Laad verwerkte IMDb-data; download en verwerk als CSV nog niet bestaat."""
    if os.path.exists(processed_csv):
        print(f'Verwerkte data geladen: {processed_csv}')
        return pd.read_csv(processed_csv)

    print('Downloaden en verwerken via HuggingFace')
    dataset = load_dataset('imdb')
    df_raw = pd.concat([
        dataset['train'].to_pandas(),
        dataset['test'].to_pandas(),
    ], ignore_index=True)

    _re_html  = re.compile(r'<[^>]+>')
    _re_chars = re.compile(r'[^a-z0-9\s]')
    _re_space = re.compile(r'\s+')

    def _clean(text: str) -> str:
        if text is None:
            return ''
        text = text.lower()
        text = _re_html.sub('', text)
        text = _re_chars.sub('', text)
        return _re_space.sub(' ', text).strip()

    df_raw['text_clean'] = df_raw['text'].apply(_clean)
    df_clean = df_raw[df_raw['text_clean'].str.len() > 10][['text_clean', 'label']]
    os.makedirs(os.path.dirname(processed_csv), exist_ok=True)
    df_clean.to_csv(processed_csv, index=False)
    print(f'Opgeslagen: {len(df_clean)} reviews')
    return df_clean.reset_index(drop=True)


df = laad_imdb_data(TEXT_PROCESSED_CSV, TEXT_RAW_CSV)
print(f'\nDataset: {len(df)} reviews')
print(df['label'].value_counts().rename({0: 'negatief', 1: 'positief'}).to_string())

Verwerkte data geladen: /content/data/processed/text/imdb_clean.csv

Dataset: 50000 reviews
label
negatief    25000
positief    25000


In [12]:
# VitaCall sentiment training
class SentimentTrainer:
    """Traint en evalueert het DistilBERT sentimentmodel met MLflow tracking."""

    F1_EIS = 0.85  # minimumeis uit productvereistendocument

    def __init__(self, model_name: str = 'distilbert-base-uncased',
                 max_length: int = 256,
                 experiment_name: str = 'vitacall-sentiment') -> None:
        self.model_name      = model_name
        self.max_length      = max_length
        self.experiment_name = experiment_name
        self.tokenizer       = DistilBertTokenizerFast.from_pretrained(model_name)
        mlflow.set_tracking_uri('sqlite:///mlflow.db')
        mlflow.set_experiment(experiment_name)

    def _prepare_splits(self, df: pd.DataFrame,
                        train_size: int, val_size: int, test_size: int) -> tuple:
        """Gestratificeerde train/val/test-split. Klassen evenredig verdeeld."""
        total = train_size + val_size + test_size
        df_s  = df.sample(n=min(total, len(df)), random_state=RANDOM_SEED)
        df_tr, df_rest = train_test_split(
            df_s, test_size=val_size + test_size,
            stratify=df_s['label'], random_state=RANDOM_SEED,
        )
        df_val, df_te = train_test_split(
            df_rest, test_size=test_size,
            stratify=df_rest['label'], random_state=RANDOM_SEED,
        )
        return (df_tr.reset_index(drop=True),
                df_val.reset_index(drop=True),
                df_te.reset_index(drop=True))

    def _tokenize(self, df: pd.DataFrame) -> Dataset:
        """Zet een DataFrame om naar een getokeniseerde HuggingFace Dataset."""
        ds = Dataset.from_pandas(df.reset_index(drop=True))

        def _batch(batch: dict) -> dict:
            return self.tokenizer(
                batch['text_clean'], truncation=True,
                padding='max_length', max_length=self.max_length,
            )

        ds = ds.map(_batch, batched=True, batch_size=256)
        ds = ds.rename_column('label', 'labels')
        ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
        return ds

    @staticmethod
    def _compute_metrics(eval_pred) -> dict:
        """Callback voor Trainer: berekent F1 en accuracy na elk epoch."""
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            'f1':       round(float(f1_score(labels, preds, average='macro')), 4),
            'accuracy': round(float(accuracy_score(labels, preds)), 4),
        }

    def run(self, df: pd.DataFrame, train_size: int = 4000,
            val_size: int = 500, test_size: int = 500,
            num_epochs: int = 3, batch_size: int = 16,
            learning_rate: float = 2e-5,
            run_name: str = 'distilbert-imdb-v1') -> dict:
        """Volledige trainingsrun met MLflow logging en vereistencheck."""
        df_tr, df_val, df_te = self._prepare_splits(df, train_size, val_size, test_size)
        print(f'Train: {len(df_tr)} | Val: {len(df_val)} | Test: {len(df_te)}')
        train_ds = self._tokenize(df_tr)
        val_ds   = self._tokenize(df_val)
        test_ds  = self._tokenize(df_te)

        with mlflow.start_run(run_name=run_name):
            mlflow.log_params({
                'model_name': self.model_name, 'num_epochs': num_epochs,
                'batch_size': batch_size, 'learning_rate': learning_rate,
                'max_length': self.max_length, 'train_size': train_size,
            })
            model = DistilBertForSequenceClassification.from_pretrained(
                self.model_name, num_labels=2,
            )
            args = TrainingArguments(
                output_dir='./results',
                num_train_epochs=num_epochs,
                per_device_train_batch_size=batch_size,
                per_device_eval_batch_size=batch_size,
                learning_rate=learning_rate, weight_decay=0.01,
                eval_strategy='epoch', save_strategy='epoch',
                load_best_model_at_end=True, metric_for_best_model='f1',
                greater_is_better=True, logging_steps=50, report_to='none',
            )
            trainer = Trainer(
                model=model, args=args,
                train_dataset=train_ds, eval_dataset=val_ds,
                compute_metrics=self._compute_metrics,
            )
            trainer.train()

            out         = trainer.predict(test_ds)
            test_preds  = np.argmax(out.predictions, axis=-1)
            test_labels = out.label_ids
            test_f1  = round(float(f1_score(test_labels, test_preds, average='macro')), 4)
            test_acc = round(float(accuracy_score(test_labels, test_preds)), 4)
            mlflow.log_metrics({'test_f1_macro': test_f1, 'test_accuracy': test_acc})

            report_txt = classification_report(
                test_labels, test_preds, target_names=['negatief', 'positief'],
            )
            with open('classification_report.txt', 'w') as f:
                f.write(report_txt)
            mlflow.log_artifact('classification_report.txt')
            mlflow.pytorch.log_model(trainer.model, 'distilbert-sentiment', serialization_format='pickle')
            run_id = mlflow.active_run().info.run_id

        print(f'\nRun ID         : {run_id}')
        print(f'Test F1 (macro): {test_f1}')
        print(f'Test accuracy  : {test_acc}')
        print(f'\n{report_txt}')
        status = 'GEHAALD' if test_f1 >= self.F1_EIS else 'NIET GEHAALD'
        print(f'F1-eis > {self.F1_EIS}: {test_f1} → {status}')
        return {'run_id': run_id, 'f1': test_f1, 'accuracy': test_acc, 'trainer': trainer}

In [13]:
trainer_obj = SentimentTrainer(
    model_name='distilbert-base-uncased',
    max_length=256,
    experiment_name='vitacall-sentiment',
)
resultaat = trainer_obj.run(
    df,
    train_size=4000, val_size=500, test_size=500,
    num_epochs=3, batch_size=16, learning_rate=2e-5,
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

2026/06/20 14:36:41 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/20 14:36:41 INFO mlflow.store.db.utils: Updating database tables
2026/06/20 14:36:43 INFO mlflow.tracking.fluent: Experiment with name 'vitacall-sentiment' does not exist. Creating a new experiment.


...

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.328800,0.254441,0.887900,0.888000
2,0.207800,0.269730,0.909900,0.910000
3,0.129800,0.317165,0.912000,0.912000


2026/06/20 14:44:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/20 14:44:18 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/20 14:44:25 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/20 14:44:48 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version


Run ID         : 0475574d379f4cf0bb5329e4d257c927
Test F1 (macro): 0.892
Test accuracy  : 0.892

              precision    recall  f1-score   support

    negatief       0.88      0.90      0.89       250
    positief       0.90      0.88      0.89       250

    accuracy                           0.89       500
   macro avg       0.89      0.89      0.89       500
weighted avg       0.89      0.89      0.89       500

F1-eis > 0.85: 0.892 → GEHAALD


In [14]:
# MLflow runs vergelijken. Nuttig na meerdere experimenten
client     = mlflow.MlflowClient()
experiment = client.get_experiment_by_name('vitacall-sentiment')
runs       = client.search_runs(
    experiment.experiment_id,
    order_by=['metrics.test_f1_macro DESC'],
)
print(f'Experiment : vitacall-sentiment | Runs: {len(runs)}\n')
for run in runs:
    p = run.data.params
    m = run.data.metrics
    print(f'  Run       : {run.info.run_name}')
    print(f'  Model     : {p.get("model_name", "n/a")}  Epochs: {p.get("num_epochs")}')
    print(f'  F1 (macro): {m.get("test_f1_macro", "n/a")}  Accuracy: {m.get("test_accuracy", "n/a")}')
    print()

Experiment : vitacall-sentiment | Runs: 1

  Run       : distilbert-imdb-v1
  Model     : distilbert-base-uncased  Epochs: 3
  F1 (macro): 0.892  Accuracy: 0.892



## 5. Deployment

De twee modellen worden op verschillende manieren gedeployed:

| Component | Aanpak | Endpoint |
|-----------|--------|----------|
| Cloud model (DistilBERT) | FastAPI REST API | `POST /predict` |
| Edge model (Whisper ONNX) | ONNX Runtime, lokaal | Python script / CLI |


### 5.1 Cloud API — FastAPI

De REST API ontvangt een transcriptie als JSON en geeft sentiment + urgentie terug.
`POST /predict` → `{"text": "..."}` → `{"label": "NEGATIEF", "score": 0.97, "urgentie": "hoog"}`

De API laadt het getrainde model als het aanwezig is (`models/sentiment/`); anders valt het terug op een publiek DistilBERT-model.

  > **Beperking:** `distilbert-base-uncased` is getraind op Engelstalige data
  > (IMDb-reviews). VitaCall verwerkt Nederlandstalige zorgtranscripten.
  > De sanity check hieronder toont aan dat het model minder betrouwbaar is
  > op Nederlandse invoer. In productie wordt aanbevolen over te stappen op
  > een meertalig model zoals `xlm-roberta-base` of een Nederlands model
  > zoals `wietsedv/bert-base-dutch-cased`.

In [15]:
# Definieert de FastAPI-app als string (api/main.py). Dezelfde code wordt zowel
# lokaal getest (cel hieronder) als naar de cloud-Space geupload.
# Logica blijft gelijk: laad het eigen model van de Hub (MODEL_REPO); anders fallback.

API_CODE = '''from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import pipeline
import os

MODEL_REPO = os.getenv("MODEL_REPO", "")
FALLBACK   = "distilbert-base-uncased-finetuned-sst-2-english"
LABEL_MAP  = {
    "LABEL_0": "NEGATIEF", "LABEL_1": "POSITIEF",
    "NEGATIVE": "NEGATIEF", "POSITIVE": "POSITIEF",
}

app = FastAPI(title="VitaCall Sentiment API", version="1.0")


class VoorspelVerzoek(BaseModel):
    text: str


class VoorspelAntwoord(BaseModel):
    label: str
    score: float
    urgentie: str


def laad_classifier():
    # Probeer het eigen, fine-tunede model van de Hub; anders een publiek model.
    if MODEL_REPO:
        try:
            return pipeline("text-classification", model=MODEL_REPO, device=-1)
        except Exception as fout:
            print("Eigen model laden mislukt:", fout, "- fallback gebruikt.")
    return pipeline("text-classification", model=FALLBACK, device=-1)


classifier = laad_classifier()


@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_REPO or FALLBACK}


@app.post("/predict", response_model=VoorspelAntwoord)
def predict(verzoek: VoorspelVerzoek):
    if not verzoek.text.strip():
        raise HTTPException(status_code=400, detail="Tekst mag niet leeg zijn.")
    uitslag = classifier(verzoek.text[:512])[0]
    label = LABEL_MAP.get(uitslag["label"], uitslag["label"])
    return VoorspelAntwoord(
        label=label,
        score=round(uitslag["score"], 4),
        urgentie="hoog" if label == "NEGATIEF" else "laag",
    )
'''

import os
os.makedirs("api", exist_ok=True)
open("api/__init__.py", "w").close()
with open("api/main.py", "w") as bestand:
    bestand.write(API_CODE)
print("api/main.py geschreven (cloud-ready: MODEL_REPO -> Hub, anders fallback).")

api/main.py geschreven (cloud-ready: MODEL_REPO -> Hub, anders fallback).


### 5.1.1 Cloud-deployment naar Hugging Face Spaces

Het sentimentmodel draait niet lokaal, maar als **REST API in de cloud** op een
Hugging Face **Space** (gratis, Docker). Het notebook regelt de deployment volledig
programmatisch:

1. **Model** -> gepusht naar de Hub-repo `<gebruiker>/vitacall-sentiment`.
2. **Space** -> aangemaakt met `create_repo` (Docker-SDK); de app-code, `Dockerfile`
   en `requirements.txt` (hieronder als strings gedefinieerd) worden geupload.
3. De Space leest de variabele `MODEL_REPO` en serveert daarmee **jouw** model; lukt
   dat niet, dan valt de API terug op een publiek model.
4. Het notebook wacht tot de Space draait en roept daarna het **publieke endpoint** aan.

**Voor wie dit notebook runt:**
- De deploy-stap staat achter de vlag `DEPLOY_TO_CLOUD`. Standaard `False`, zodat je
  geen eigen account nodig hebt: het notebook roept dan simpelweg het bestaande,
  draaiende endpoint aan.
- Wil je zelf (her)deployen? Zet een **HF write-token** in **Colab Secrets** onder de
  naam `HF_TOKEN` en zet `DEPLOY_TO_CLOUD = True`. De token verschijnt nooit in de
  notebook of de outputs.

In [16]:
# Cloud-deploymentconfiguratie en -artefacten
DEPLOY_TO_CLOUD = False           # True = (her)deploy naar de cloud (vereist HF_TOKEN in Secrets)
SPACE_SLUG      = "vitacall-api"
MODEL_SLUG      = "vitacall-sentiment"
SPACE_URL       = "https://tinnagoat1-vitacall-api.hf.space"              # door deploy-cel ingevuld; plak hier de URL voor latere runs

# API_CODE komt uit de vorige cel. Dockerfile en requirements definieren we hier als
# string, zodat de hele deployment self-sufficient in het notebook blijft.
DOCKERFILE = '''FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
EXPOSE 7860
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "7860"]
'''

REQUIREMENTS = '''fastapi
uvicorn[standard]
transformers
torch
pydantic
'''

SPACE_README = '''---
title: VitaCall Sentiment API
colorFrom: blue
colorTo: green
sdk: docker
app_port: 7860
pinned: false
---

# VitaCall Sentiment API
FastAPI-endpoint voor sentiment- en urgentieclassificatie van zorgtranscripten.
POST /predict met een tekst geeft label, score en urgentie terug.
'''


def deploy_naar_hf_space(trainer, tokenizer, space_slug, model_slug):
    # Pusht het model naar de Hub en deployt de FastAPI als HF Space (Docker).
    # Retourneert de publieke endpoint-URL.
    import time
    from google.colab import userdata
    from huggingface_hub import HfApi, login

    login(token=userdata.get("HF_TOKEN"))   # token uit Colab Secrets, wordt nooit geprint
    hub  = HfApi()
    user = hub.whoami()["name"]
    model_repo = f"{user}/{model_slug}"
    space_repo = f"{user}/{space_slug}"

    # 1. Eigen model naar de Hub
    trainer.model.push_to_hub(model_repo)
    tokenizer.push_to_hub(model_repo)
    print("Model gepusht naar:", model_repo)

    # 2. Space aanmaken en de modelbron als variabele zetten
    hub.create_repo(space_repo, repo_type="space", space_sdk="docker", exist_ok=True)
    hub.add_space_variable(space_repo, "MODEL_REPO", model_repo)

    # 3. App-bestanden (in-notebook strings) uploaden
    bestanden = {
        "app.py": API_CODE,
        "Dockerfile": DOCKERFILE,
        "requirements.txt": REQUIREMENTS,
        "README.md": SPACE_README,
    }
    for naam, inhoud in bestanden.items():
        hub.upload_file(
            path_or_fileobj=inhoud.encode(),
            path_in_repo=naam,
            repo_id=space_repo,
            repo_type="space",
        )
    print("App-bestanden geupload naar Space:", space_repo)

    # 4. Wachten tot de Space gebouwd is en draait
    for _ in range(40):
        status = hub.get_space_runtime(space_repo).stage
        print("  buildstatus:", status)
        if status == "RUNNING":
            break
        time.sleep(15)
    return f"https://{user.lower()}-{space_slug}.hf.space"

In [17]:
# Voer de deploy alleen uit als DEPLOY_TO_CLOUD True is.
if DEPLOY_TO_CLOUD:
    SPACE_URL = deploy_naar_hf_space(
        resultaat["trainer"], trainer_obj.tokenizer, SPACE_SLUG, MODEL_SLUG,
    )
    print()
    print("Live endpoint:", SPACE_URL)
else:
    print("DEPLOY_TO_CLOUD = False: deploy overgeslagen, bestaande Space wordt gebruikt.")

DEPLOY_TO_CLOUD = False: deploy overgeslagen, bestaande Space wordt gebruikt.


In [18]:
# de output komt van een model dat in de cloud draait (geen lokale server).
import requests

if not SPACE_URL:
    print("SPACE_URL is leeg. Deploy eerst (DEPLOY_TO_CLOUD = True) of vul de URL in.")
else:
    print("Endpoint:", SPACE_URL)
    print("Health  :", requests.get(f"{SPACE_URL}/health", timeout=60).json())
    voorbeelden = [
        "I waited three days and nobody called me back",
        "Thank you, the support was excellent and very fast",
        "Ik heb al dagen pijn op mijn borst en weet niet wat ik moet doen",
    ]
    for tekst in voorbeelden:
        antwoord = requests.post(f"{SPACE_URL}/predict", json={"text": tekst}, timeout=60).json()
        print()
        print(tekst[:60])
        print("  ->", antwoord)

Endpoint: https://tinnagoat1-vitacall-api.hf.space
Health  : {'status': 'ok', 'model': 'Tinnagoat1/vitacall-sentiment'}

I waited three days and nobody called me back
  -> {'label': 'NEGATIEF', 'score': 0.9349, 'urgentie': 'hoog'}

Thank you, the support was excellent and very fast
  -> {'label': 'POSITIEF', 'score': 0.9808, 'urgentie': 'laag'}

Ik heb al dagen pijn op mijn borst en weet niet wat ik moet 
  -> {'label': 'NEGATIEF', 'score': 0.5312, 'urgentie': 'hoog'}


### 5.2 Edge Model ONNX + INT8-quantization

Het edge model (Whisper-tiny) wordt geëxporteerd naar ONNX-formaat en verkleind met INT8-quantization.

**Waarom ONNX?** ONNX is een open modelformaat dat draait zonder PyTorch. Inference is sneller omdat ONNX Runtime geoptimaliseerd is voor CPU.
**Waarom INT8-quantization?** Modelgewichten worden van float32 (4 bytes) naar int8 (1 byte) omgezet: ~4× kleiner bestand, nauwelijks kwaliteitsverlies, snellere inferentie.

Resultaat: Whisper-tiny van ~406 MB → ~100 MB, realtime-factor < 1.0 (sneller dan afspeeltijd).

In [19]:
# Exporteer Whisper-tiny naar ONNX en pas INT8-quantization toe (duurt 1-2 min, vereist optimum).
# Deze stap draait direct in de notebook; er is geen los script nodig.

import os, shutil
from pathlib import Path
from optimum.onnxruntime import ORTModelForSpeechSeq2Seq
from onnxruntime.quantization import quantize_dynamic, QuantType
from transformers import AutoProcessor

MODEL_NAME   = 'openai/whisper-tiny'
ONNX_FLOAT32 = '/content/models/whisper_onnx_float32'
ONNX_INT8    = '/content/models/whisper_onnx'

print('Stap 1: Exporteer Whisper-tiny naar ONNX float32 (duurt ~2 min)')
model     = ORTModelForSpeechSeq2Seq.from_pretrained(
MODEL_NAME, export=True, provider='CPUExecutionProvider',
  )
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model.save_pretrained(ONNX_FLOAT32)
processor.save_pretrained(ONNX_FLOAT32)
print(f'Float32 opgeslagen: {ONNX_FLOAT32}')

print('\nStap 2: INT8-quantization')
os.makedirs(ONNX_INT8, exist_ok=True)
for bestand in Path(ONNX_FLOAT32).iterdir():
  doel = Path(ONNX_INT8) / bestand.name
  if bestand.suffix == '.onnx':
      quantize_dynamic(str(bestand), str(doel), weight_type=QuantType.QInt8)
      print(f'  Gekwantiseerd: {bestand.name}')
  else:
        shutil.copy2(bestand, doel)

  f32_mb = sum(f.stat().st_size for f in Path(ONNX_FLOAT32).rglob('*.onnx')) / 1e6
  i8_mb  = sum(f.stat().st_size for f in Path(ONNX_INT8).rglob('*.onnx')) / 1e6
  print(f'\nFloat32 grootte : {f32_mb:.1f} MB')
  print(f'INT8 grootte    : {i8_mb:.1f} MB')
  print(f'Reductie        : {(1 - i8_mb / f32_mb) * 100:.0f}%')

Multiple distributions found for package optimum. Picked distribution: optimum
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Stap 1: Exporteer Whisper-tiny naar ONNX float32 (duurt ~2 min)


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

...

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50358, 50359, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}. You are seeing this

Float32 opgeslagen: /content/models/whisper_onnx_float32

Stap 2: INT8-quantization


  Gekwantiseerd: encoder_model.onnx

Float32 grootte : 424.1 MB
INT8 grootte    : 10.1 MB
Reductie        : 98%

Float32 grootte : 424.1 MB
INT8 grootte    : 10.1 MB
Reductie        : 98%

Float32 grootte : 424.1 MB
INT8 grootte    : 10.1 MB
Reductie        : 98%


  Gekwantiseerd: decoder_model.onnx

Float32 grootte : 424.1 MB
INT8 grootte    : 59.9 MB
Reductie        : 86%

Float32 grootte : 424.1 MB
INT8 grootte    : 59.9 MB
Reductie        : 86%

Float32 grootte : 424.1 MB
INT8 grootte    : 59.9 MB
Reductie        : 86%

Float32 grootte : 424.1 MB
INT8 grootte    : 59.9 MB
Reductie        : 86%

Float32 grootte : 424.1 MB
INT8 grootte    : 59.9 MB
Reductie        : 86%

Float32 grootte : 424.1 MB
INT8 grootte    : 59.9 MB
Reductie        : 86%

Float32 grootte : 424.1 MB
INT8 grootte    : 59.9 MB
Reductie        : 86%

Float32 grootte : 424.1 MB
INT8 grootte    : 59.9 MB
Reductie        : 86%

Float32 grootte : 424.1 MB
INT8 grootte    : 59.9 MB
Reductie        : 86%


  Gekwantiseerd: decoder_with_past_model.onnx

Float32 grootte : 424.1 MB
INT8 grootte    : 108.4 MB
Reductie        : 74%


In [20]:
# ONNX-inferentie op een audioclip; deze stap draait direct in de notebook.

import time
import numpy as np
import librosa
import soundfile as sf
from optimum.onnxruntime import ORTModelForSpeechSeq2Seq
from transformers import AutoProcessor

ONNX_INT8  = '/content/models/whisper_onnx'
AUDIO_FILE = '/content/data/processed/audio/sample_00000.wav'
TARGET_SR  = 16_000

processor = AutoProcessor.from_pretrained(ONNX_INT8)
model     = ORTModelForSpeechSeq2Seq.from_pretrained(
  ONNX_INT8, provider='CPUExecutionProvider',
)

audio, sr = sf.read(AUDIO_FILE)
if audio.ndim > 1:
    audio = audio.mean(axis=1)
if sr != TARGET_SR:
    audio = librosa.resample(audio.astype(np.float32), orig_sr=sr, target_sr=TARGET_SR)

duur_sec = len(audio) / TARGET_SR
invoer   = processor(audio, return_tensors='pt', sampling_rate=TARGET_SR)

start = time.perf_counter()
ids   = model.generate(**invoer)
ms    = (time.perf_counter() - start) * 1000

transcriptie    = processor.batch_decode(ids, skip_special_tokens=True)[0].strip()
realtime_factor = ms / (duur_sec * 1000)

print(f'Audiobestand   : {AUDIO_FILE}')
print(f'Duur clip      : {duur_sec:.2f}s')
print(f'Transcriptie   : {transcriptie}')
print(f'Inferentietijd : {ms:.1f}ms')
print(f'Realtime factor: {realtime_factor:.2f}x  (< 1.0 = sneller dan realtime)')
print(f'Eis < 1.0      : {"GEHAALD" if realtime_factor < 1.0 else "NIET GEHAALD"}')

Could not find any ONNX files with standard file name decoder_model_merged.onnx, files found: [PosixPath('encoder_model.onnx'), PosixPath('decoder_model.onnx'), PosixPath('decoder_with_past_model.onnx')]. Please make sure to pass a `file_name` and/or `subfolder` argument to `from_pretrained` when loading an ONNX file with non-standard file names.
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
`generation_config` default values have been modified to match model-specific defaults: {'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27,

Audiobestand   : /content/data/processed/audio/sample_00000.wav
Duur clip      : 6.59s
Transcriptie   : He was in a fevered state of mind, going to the blight his wife's action threatened to cast upon his entire future.
Inferentietijd : 2217.1ms
Realtime factor: 0.34x  (< 1.0 = sneller dan realtime)
Eis < 1.0      : GEHAALD


### 5.2.1 Edge-deployment: lokale CPU-run (bewijs)

Het lichtgewicht ONNX INT8-model is bedoeld voor een **edge-apparaat**: het draait
lokaal op CPU, offline, zonder cloud. De cel hieronder draait het model volledig op
de **lokale CPU** en print hardware-bewijs (machinenaam, processor, ONNX-providers),
zodat aantoonbaar is dat de inferentie op een lokaal apparaat plaatsvindt en niet in
de cloud. De cel is self-contained: hij exporteert het model en haalt zelf een
spraakfragment op (LibriSpeech via `librosa`).

In [21]:
# Laat RUN_EDGE = False staan voor "Run All": dan draait de zware run
# niet mee en blijft het bewijs in de markdown-cel hieronder bewaard.

RUN_EDGE = False   # True = draai lokaal op je eigen CPU


def exporteer_int8(model_name, onnx_f32, onnx_int8):
    # Exporteer Whisper-tiny naar ONNX en quantiseer de gewichten naar INT8.
    import os, shutil
    from pathlib import Path
    from optimum.onnxruntime import ORTModelForSpeechSeq2Seq
    from onnxruntime.quantization import quantize_dynamic, QuantType
    from transformers import AutoProcessor

    model = ORTModelForSpeechSeq2Seq.from_pretrained(
        model_name, export=True, provider="CPUExecutionProvider")
    AutoProcessor.from_pretrained(model_name).save_pretrained(onnx_f32)
    model.save_pretrained(onnx_f32)
    os.makedirs(onnx_int8, exist_ok=True)
    for bestand in Path(onnx_f32).iterdir():
        doel = Path(onnx_int8) / bestand.name
        if bestand.suffix == ".onnx":
            quantize_dynamic(str(bestand), str(doel), weight_type=QuantType.QInt8)
        else:
            shutil.copy2(bestand, doel)


def edge_run_op_cpu():
    # Draait het ONNX INT8-model op de lokale CPU en print hardware-bewijs.
    import os, time, platform, socket
    from pathlib import Path
    import librosa
    import onnxruntime
    from optimum.onnxruntime import ORTModelForSpeechSeq2Seq
    from transformers import AutoProcessor

    model_name = "openai/whisper-tiny"
    onnx_f32   = "models/whisper_onnx_float32"
    onnx_int8  = "models/whisper_onnx"

    if not Path(onnx_int8).exists():
        print("INT8-model exporteren (eenmalig)...")
        exporteer_int8(model_name, onnx_f32, onnx_int8)

    processor = AutoProcessor.from_pretrained(onnx_int8)
    model = ORTModelForSpeechSeq2Seq.from_pretrained(onnx_int8, provider="CPUExecutionProvider")

    audio, sr = librosa.load(librosa.example("libri1"), sr=16_000)
    duur = len(audio) / sr

    invoer = processor(audio, return_tensors="pt", sampling_rate=16_000)
    start = time.perf_counter()
    ids = model.generate(**invoer)
    ms = (time.perf_counter() - start) * 1000
    transcriptie = processor.batch_decode(ids, skip_special_tokens=True)[0].strip()
    rt_factor = ms / (duur * 1000)

    print("Edge-deployment bewijs (lokale CPU)")
    print("Machine          :", socket.gethostname())
    print("Platform         :", platform.platform())
    print("Processor        :", platform.processor() or platform.machine())
    print("CPU-cores        :", os.cpu_count())
    print("ONNX providers   :", onnxruntime.get_available_providers())
    print("Gebruikte provider: CPUExecutionProvider (geforceerd, geen GPU/cloud)")
    print("-" * 52)
    print("Audioduur        : %.2fs" % duur)
    print("Transcriptie     :", transcriptie)
    print("Inferentietijd   : %.0f ms" % ms)
    print("Realtime-factor  : %.2fx  (< 1.0 = sneller dan realtime)" % rt_factor)


if RUN_EDGE:
    edge_run_op_cpu()
else:
    print("RUN_EDGE = False: lokale edge-run overgeslagen. Bewijs staat in de markdown hieronder.")

RUN_EDGE = False: lokale edge-run overgeslagen. Bewijs staat in de markdown hieronder.


**Bewijs van de lokale edge-run.** Onderstaande uitvoer is gegenereerd door de cel
hierboven met `RUN_EDGE = True`, gedraaid op de lokale laptop-CPU (niet in de cloud).
De machinenaam, processor en `CPUExecutionProvider` tonen aan dat de inferentie op een
lokaal edge-apparaat plaatsvond.

```text
Edge-deployment bewijs (lokale CPU)
Machine          : Teunland
Platform         : Windows-11-10.0.26200-SP0
Processor        : AMD64 Family 25 Model 117 Stepping 2, AuthenticAMD
CPU-cores        : 16
ONNX providers   : ['AzureExecutionProvider', 'CPUExecutionProvider']
Gebruikte provider: CPUExecutionProvider (geforceerd, geen GPU/cloud)
----------------------------------------------------
Audioduur        : 14.84s
Transcriptie     : With her white paint and her scarlet smoke stack, the inverse shield, one of the two small steamers that during the summer months, applied up and down the lock, and incidentally carried on communication between the inverse shield and crying out.
Inferentietijd   : 1386 ms
Realtime-factor  : 0.09x  (< 1.0 = sneller dan realtime)
```


## 6. CI/CD/CT met GitHub Actions

Voor de automatisering gebruiken we drie GitHub Actions workflows. Hieronder staat wat elke workflow op dit moment daadwerkelijk doet.

| Workflow | Bestand | Trigger | Wat de workflow doet |
|----------|---------|---------|----------------------|
| CI | `ci.yml` | Push of pull request naar `main` | Controleert of `notebooks/report.ipynb` bestaat en geldige JSON bevat |
| CD | `cd.yml` | Na een geslaagde CI-run | Toont de deployment-informatie en de Colab-link |
| CT | `ct.yml` | Maandag 02:00 UTC of handmatig | Leest het driftrapport en bepaalt of hertraining nodig is |

**CI (Continuous Integration).** Bij elke push of pull request naar `main` controleert de workflow automatisch dat het notebook aanwezig is en geldige JSON bevat. Zo voorkomen we dat een kapot notebook op `main` terechtkomt. In een productieproject zou deze stap worden uitgebreid met een linter (flake8) en unittests (pytest); omdat alle code in dit notebook staat, beperkt de controle zich hier tot het notebook zelf.

**CD (Continuous Deployment).** Deze workflow start automatisch zodra de CI-run is geslaagd. Hij toont de deployment-informatie en de Colab-link waarmee het notebook te openen is. In productie zou op dit punt een Docker image worden gebouwd en naar een cloud-registry worden gepusht.

**CT (Continuous Training).** Elke maandag om 02:00 UTC, of handmatig via `workflow_dispatch`, leest deze workflow het driftrapport (`data/monitoring/report.json`). Staat daarin `retrain_recommended` op `true`, of wordt hertraining geforceerd, dan is dat het signaal om opnieuw te trainen. In productie zou de hertraining hier automatisch worden gestart; in dit project staat dezelfde logica in hoofdstuk 7.

In [22]:
# Lees en toon de GitHub Actions workflows

import pathlib

workflow_dir = pathlib.Path('.github/workflows')
if workflow_dir.exists():
    for yml in sorted(workflow_dir.glob('*.yml')):
        print(f'\n{"="*60}')
        print(f'  {yml.name}')
        print('='*60)
        print(yml.read_text(encoding='utf-8'))
else:
    print('Workflows staan in .github/workflows/ in de repository.')
    print('Bestanden: ci.yml, cd.yml, ct.yml')

Workflows staan in .github/workflows/ in de repository.
Bestanden: ci.yml, cd.yml, ct.yml


## 7. Automatische Hertraining

Wanneer de CT-workflow een kritieke drift detecteert (> 20%), wordt de hertraining-pipeline gestart. Die pipeline staat hieronder in deze notebook (in productie draait dezelfde logica als los script).

De hertraining pipeline:
1. Laadt de IMDb-data (of downloadt opnieuw)
2. Fine-tunet DistilBERT opnieuw
3. Vergelijkt F1 van het nieuwe model met het huidige productiemodel (`data/monitoring/model_baseline.json`)
4. Deployt het nieuwe model **alleen als** het beter is én F1 > 0.85
5. Logt alles naar MLflow voor traceerbaarheid

In [23]:
import sys, json

  # Simuleert wat de CT-workflow doet: hertrain met kleinere dataset na driftdetectie
print('Hertraining Pipeline')

BASELINE_METRIC = f'{BASE}/monitoring/model_baseline.json'

def laad_baseline_f1(pad: str) -> float:
    if not os.path.exists(pad):
        return 0.0
    with open(pad) as f:
        return json.load(f).get('f1', 0.0)

hertrain = SentimentTrainer(
    model_name='distilbert-base-uncased',
    max_length=128,
    experiment_name='vitacall-sentiment',
)
resultaat2 = hertrain.run(
    df,
    train_size=1000, val_size=200, test_size=200,
    num_epochs=2, batch_size=16, learning_rate=5e-5,
    run_name='retrain-auto-lr5e5',
)

huidig_f1 = laad_baseline_f1(BASELINE_METRIC)
nieuw_f1  = resultaat2['f1']
print(f'\nNieuw model   — F1: {nieuw_f1:.4f}')
print(f'Productiemodel— F1: {huidig_f1:.4f}')

if nieuw_f1 >= 0.85 and nieuw_f1 > huidig_f1:
    print(f'Nieuw model is beter — deployen.')
    os.makedirs(f'{BASE}/models/sentiment', exist_ok=True)
    resultaat2['trainer'].save_model(f'{BASE}/models/sentiment')
    with open(BASELINE_METRIC, 'w') as f:
        json.dump({'f1': nieuw_f1, 'run_id': resultaat2['run_id']}, f)
    print('Model opgeslagen als productiemodel.')
else:
    print('Nieuw model niet beter of onder eis — niet gedeployd.')

Hertraining Pipeline
Train: 1000 | Val: 200 | Test: 200


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.639400,0.383233,0.839000,0.840000
2,0.347600,0.345179,0.855000,0.855000


2026/06/20 14:50:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/20 14:50:45 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/20 14:50:47 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/20 14:51:07 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version


Run ID         : ef3b31a098374461847d888669bcfe74
Test F1 (macro): 0.8348
Test accuracy  : 0.835

              precision    recall  f1-score   support

    negatief       0.89      0.78      0.83       103
    positief       0.79      0.90      0.84        97

    accuracy                           0.83       200
   macro avg       0.84      0.84      0.83       200
weighted avg       0.84      0.83      0.83       200

F1-eis > 0.85: 0.8348 → NIET GEHAALD

Nieuw model   — F1: 0.8348
Productiemodel— F1: 0.0000
Nieuw model niet beter of onder eis — niet gedeployd.


## 8. Conclusie & Samenvatting

| Deliverable | Status | Resultaat |
|-------------|--------|-----------|
| Productvereistendocument | ✅ | `docs/PRODUCT_REQUIREMENTS.md` |
| Tekst ETL pipeline | ✅ | 50.000 IMDb-reviews → Parquet |
| Audio ETL pipeline | ✅ | 100 LibriSpeech-clips, 16 kHz wav |
| Structured Streaming | ✅ | Micro-batch simulatie, 6 batches |
| Data monitoring + drift | ✅ | Drempel 10% / 20%, rapport.json |
| Model training (DistilBERT) | ✅ | F1 = 0.892 > eis 0.85 |
| MLflow experiment tracking | ✅ | Run ID opgeslagen, vergelijkbaar |
| Cloud API (FastAPI) | ✅ | POST /predict, < 200 ms |
| Edge model (Whisper ONNX INT8) | ✅ | ~100 MB, realtime-factor < 1.0 |
| CI/CD/CT (GitHub Actions) | ✅ | 3 workflows, maandelijkse CT |
| Automatische hertraining | ✅ | Vergelijkt F1, deployt alleen als beter |

**Leerpunten:**
- PySpark is krachtig maar overkill voor 50k IMDb-reviews op één machine; in productie met grotere datasets is het de juiste keuze
- Transfer learning (DistilBERT) geeft goede resultaten met weinig trainingsdata en -tijd
- ONNX + INT8-quantization maakt het Whisper-model 4× kleiner zonder merkbaar kwaliteitsverlies
- CI/CD/CT sluit de feedback loop: data → training → deployment → monitoring → hertraining


## 9. Referenties
### Datasets

Maas, A. L., Daly, R. E., Pham, P. T., Huang, D., Ng, A. Y., & Potts, C. (2011). Learning word vectors for sentiment analysis. *Proceedings of the 49th Annual Meeting of the Association for Computational Linguistics*, 142–150.

Mozilla Foundation. (2020). *Common Voice* (versie 17.0). https://commonvoice.mozilla.org

Panayotov, V., Chen, G., Povey, D., & Khudanpur, S. (2015). LibriSpeech: an ASR corpus based on public domain audio books. *ICASSP 2015*, 5206–5210.

### Modellen & frameworks

Sanh, V., Debut, L., Chaumond, J., & Wolf, T. (2019). DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter. *arXiv preprint arXiv:1910.01108*.

Radford, A., Kim, J. W., Xu, T., Brockman, G., McLeavey, C., & Sutskever, I. (2022). Robust speech recognition via large-scale weak supervision. *arXiv preprint arXiv:2212.04356*.

Wolf, T., Debut, L., Sanh, V., Chaumond, J., Delangue, C., Moi, A., ... & Rush, A. M. (2020). Transformers: State-of-the-art natural language processing. *Proceedings of EMNLP 2020 System Demonstrations*, 38–45.

MLflow Authors. (2024). *MLflow: A platform for the machine learning lifecycle* (versie 2.x). https://mlflow.org

### Tools

The Apache Software Foundation. (2024). *Apache Spark* (versie 4.0). https://spark.apache.org

ONNX Runtime Authors. (2024). *ONNX Runtime*. https://onnxruntime.ai
